In [ ]:
import pandas as pd
import numpy as np

PROCESSED = '../notebooks/data/processed'

X_train = pd.read_parquet(f'{PROCESSED}/X_train.parquet')
Y_train = pd.read_parquet(f'{PROCESSED}/Y_train.parquet').iloc[:, 0]
X_test  = pd.read_parquet(f'{PROCESSED}/X_test.parquet')
Y_test  = pd.read_parquet(f'{PROCESSED}/Y_test.parquet').iloc[:, 0]


print(f'X_train: {X_train.shape}')
print(f"X_test: {X_test.shape}")
print(f'Y_train fraud rate: {Y_train.mean():.3%}')
print(f'Y_test fraud rate: {Y_test.mean():.3%}')

In [ ]:
import sys
sys.path.append('../')

from src.models.train import (train_xgboost_baseline, train_lightgbm_baseline, save_model)

from src.models.evaluate import evaluate_model, log_results, compare_runs

In [ ]:
xgb_model = train_xgboost_baseline(X_train, Y_train)
lgbm_model = train_lightgbm_baseline(X_train,Y_train)

In [ ]:
xgb_results = evaluate_model(xgb_model, X_test, Y_test, model_name = "xgboost_baseline")

lgbm_results = evaluate_model(lgbm_model, X_test,Y_test, model_name = 'lgbm_baseline')

log_results(xgb_results)
log_results(lgbm_results)

compare_runs()

save_model(xgb_model, "xgboost_baseline")
save_model(lgbm_model,"lightgbm_baseline")

In [ ]:
import sys
sys.path.append('../')

from src.models.train import (tune_xgboost, save_model)

from src.models.evaluate import evaluate_model, log_results, compare_runs

In [ ]:
tuned_xgb, study = tune_xgboost(X_train, Y_train, n_trials=50)

tuned_results = evaluate_model(
    tuned_xgb, X_test, Y_test,
    model_name="xgboost_tuned",

)

log_results(tuned_results)

save_model(tuned_xgb, "xgboost_tuned")

compare_runs()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

Y_prob = tuned_xgb.predict_proba(X_test)[:,1]

precisions, recalls, thresholds = precision_recall_curve(Y_test,Y_prob)

f1_score  = 2* (precisions*recalls)/ (precisions + recalls + 1e-8)
best_idx = f1_score.argmax()
best_threshold = thresholds[best_idx]

print(f"Best threshold: {best_threshold:.4f}")
print(f"At this threshold:")
print(f"  Precision: {precisions[best_idx]:.4f}")
print(f"  Recall:    {recalls[best_idx]:.4f}")
print(f"  F1:        {f1_score[best_idx]:.4f}")


plt.figure(figsize=(10,5))

plt.subplot(1, 2, 1)
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.plot(thresholds, f1_score[:-1], label='F1')
plt.axvline(best_threshold, color='red', linestyle='--',
            label=f'Best threshold: {best_threshold:.3f}')
plt.xlabel('Threshold')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend()


plt.subplot(1, 2, 2)
plt.plot(recalls[:-1], precisions[:-1])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.axvline(recalls[best_idx], color='red', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
import os

log_path = 'data/model_results/model_log.json'
if os.path.exists(log_path):
    os.remove(log_path)
    print("Corrupted log deleted")

In [ ]:
from src.models.evaluate import evaluate_model, log_results, compare_runs

Y_prob = tuned_xgb.predict_proba(X_test)[:, 1]

tuned_results = evaluate_model(
    tuned_xgb, X_test, Y_test,
    threshold=0.5,
    model_name="xgboost_tuned"
)

tuned_results_optimal = evaluate_model(
    tuned_xgb, X_test, Y_test,
    threshold=0.2161,
    model_name="xgboost_tuned_optimal_threshold"
)

log_results(tuned_results)
log_results(tuned_results_optimal)
compare_runs()

In [ ]:
import os
import joblib

# Create directory if it doesn't exist
os.makedirs('data/models/', exist_ok=True)

# Save to notebooks/data/models/ (same pattern as processed data)
model_bundle = {
    "model": tuned_xgb,
    "threshold": best_threshold,
    "metrics": tuned_results_optimal
}

joblib.dump(model_bundle, 'data/models/xgboost_production.pkl')
print(f"Model bundle saved with threshold: {best_threshold:.4f}")